# SageMaker Feature Store
Cria o feature group e ingere as features geradas pelo Glue.

Meu bucket : s3://ny-taxi-pedro/processed/yellow/

In [12]:
#%pip install -q awswrangler sagemaker

import time
import boto3
import awswrangler as wr
import pandas as pd
from sagemaker.feature_store.feature_group import FeatureGroup
import sagemaker


BUCKET = "s3://ny-taxi-pedro/"
PROCESSED_PREFIX = "processed/yellow/"
REGION = "us-east-1"
FEATURE_GROUP = "sagemaker-nyc-taxi-tip-fg"
SAMPLE_SIZE = 200_000

session = sagemaker.Session()
role    = sagemaker.get_execution_role()
print("Role:", role)
print("Region:", session.boto_region_name)

Role: arn:aws:iam::633203421113:role/service-role/AmazonSageMaker-ExecutionRole-20260630T194736
Region: us-east-1


In [13]:
# Ler os dados e processa amostra
df = wr.s3.read_parquet(f"{BUCKET}{PROCESSED_PREFIX}")
print(f"Linhas totais: {len(df)}")
print("Distribuição do label high_tip:")
print(df["high_tip"].value_counts(normalize=True))

if len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print("Linhas para ingestão:", len(df))
df.head()

Linhas totais: 7319048
Distribuição do label high_tip:
high_tip
1    0.754044
0    0.245956
Name: proportion, dtype: Float64
Linhas para ingestão: 200000


,ride_id,event_time,trip_distance,trip_duration_min,passenger_count,pickup_hour,pickup_dayofweek,is_weekend,PULocationID,DOLocationID,fare_amount,high_tip
0,f558683622c6e8503cfe2558fc434c6abb2d010bb75b87...,2025-01-16T23:15:02Z,0.61,3.266667,1,23,5,0,249,125,5.8,1
1,bd258f4bdfe4390e3b95d004795019220d9e43d971489c...,2025-02-26T15:56:41Z,1.63,10.266667,1,15,4,0,170,237,12.1,1
2,380c4f24396005b5f52b4a9ab025968c82713952c38ae4...,2025-03-15T16:58:46Z,0.72,5.900000,1,16,7,1,236,237,7.2,1
3,15495886d33f8251793f28c981846eaee736ea8cceece5...,2025-02-15T00:56:15Z,0.28,5.083333,1,0,7,1,114,148,6.5,1
4,7d8149fdb640b02c1b5e9db67e8a143827bcad90395d62...,2025-02-26T19:50:32Z,1.70,10.816667,1,19,4,0,142,68,12.1,0


In [14]:
# Corrigindo os tipos no dataframe, o FeatureStore exige tipos corretos

df['ride_id'] = df['ride_id'].astype(str)
df['event_time'] = df['event_time'].astype(str)

int_cols = [
    "passenger_count", "pickup_hour", "pickup_dayofweek",
    "is_weekend", "PULocationID", "DOLocationID", "high_tip"
]

for col in int_cols:
    df[col] = df[col].astype("int64")
    float_cols = ["trip_distance", "trip_duration_min", "fare_amount"]
for col in float_cols:
    df[col] = df[col].astype("float64")

df.dtypes


ride_id               object
event_time            object
trip_distance        float64
trip_duration_min    float64
passenger_count        int64
pickup_hour            int64
pickup_dayofweek       int64
is_weekend             int64
PULocationID           int64
DOLocationID           int64
fare_amount          float64
high_tip               int64
dtype: object

In [17]:
# Cria o Feature Group  online e offline
feature_group = FeatureGroup(name=FEATURE_GROUP, sagemaker_session=session)
feature_group.load_feature_definitions(data_frame=df)

OFFLINE_S3 = f"{BUCKET}feature-store/"
print(repr(OFFLINE_S3))
feature_group.create(
    s3_uri=OFFLINE_S3,
    record_identifier_name="ride_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True,
)
print("Criando feature group...")

's3://ny-taxi-pedro/feature-store/'
Criando feature group...


In [18]:
# Espera o Feature Group ficar 'Created'
def wait_fg(fg):
    status = fg.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("...", status)
        time.sleep(10)
        status = fg.describe().get("FeatureGroupStatus")
    print("Status final:", status)

wait_fg(feature_group)


... Creating
Status final: Created


In [19]:
# Ingestão (popula online store imediatamente; offline store leva alguns minutos)
print('Populando feature stores')
feature_group.ingest(data_frame=df, max_workers=2, wait=True)
print("População concluída.")

Populando feature stores
População concluída.


In [20]:
# Passa variáveis para os próximos notebooks
%store FEATURE_GROUP
%store BUCKET
%store REGION
# guarda alguns ride_ids para testar inferência
sample_ids = df["ride_id"].head(5).tolist()
%store sample_ids
print("Salvo:", FEATURE_GROUP, BUCKET, REGION)
print("IDs de teste:", sample_ids)


Stored 'FEATURE_GROUP' (str)
Stored 'BUCKET' (str)
Stored 'REGION' (str)
Stored 'sample_ids' (list)
Salvo: sagemaker-nyc-taxi-tip-fg s3://ny-taxi-pedro/ us-east-1
IDs de teste: ['f558683622c6e8503cfe2558fc434c6abb2d010bb75b878cebb112fb447c958c', 'bd258f4bdfe4390e3b95d004795019220d9e43d971489cb537ee05124a00ac80', '380c4f24396005b5f52b4a9ab025968c82713952c38ae43937044f727f61b12a', '15495886d33f8251793f28c981846eaee736ea8cceece504fc67fb757b116ae3', '7d8149fdb640b02c1b5e9db67e8a143827bcad90395d62db841b0ae7fefc088e']


In [ ]:
# Exclusao do feature group antigo:
# Para corrigir S3 inválido, tive problemas pipipipi
feature_group.delete()